# Graphene electrostatics and quantum capacitance

**Professor Nano Labs · Crystals & energy bands**

**Try it live in your browser:** [Graphene electrostatics lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/graphene-electrostatics/)
**Watch first:** [Calculating electron and hole densities in graphene](https://www.youtube.com/watch?v=wBRvi_NRIog), then [Electrostatics of graphene and quantum capacitance](https://www.youtube.com/watch?v=4bVyZTcIv3w)
**Watch later:** [Calculating the band structure of graphene](https://www.youtube.com/watch?v=3b34QM15JTM) · [Deriving the density-of-states for any dispersion and dimensionality](https://www.youtube.com/watch?v=vNBttmWca_U) · [Fermi Dirac vs Bose Einstein vs Classical Maxwell Boltzmann distributions in 3 minutes!](https://www.youtube.com/watch?v=e9TQs-EDiUs)

Graphene's bands near the Dirac points are cones, $E=\pm\hbar v_F|\mathbf k|$, with $v_F=10^6$ m/s as in the video and four states per $\mathbf k$ (spin and valley). Then
$$D(E)=\frac{2|E|}{\pi\hbar^2v_F^2},\qquad n=\int_0^\infty fD\,dE,\qquad p=\int_{-\infty}^0(1-f)D\,dE,\qquad e(n-p)=C_{ox}\Big(V-\frac{E_F}{e}\Big),\qquad \frac1{C_{tot}}=\frac1{C_{ox}}+\frac1{C_{qm}}.$$
We work with energies in eV, lengths in nm and densities per nm² ($1$ nm⁻² $=10^{14}$ cm⁻²). The three sections match the lab's three tabs. It needs NumPy, SciPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq
from scipy.special import spence
from scipy.constants import e, hbar, k as kB, epsilon_0

vF = 1e6
hv = hbar * vF / e * 1e9                     # hbar vF in eV nm
c = 2 / (np.pi * hv ** 2)                    # D(E) = c|E|, states per eV per nm^2
atoms = 4 / (3 * np.sqrt(3) * 0.142 ** 2)    # carbon atoms per nm^2
kT = lambda T: kB * T / e                    # eV
cm2 = lambda x: x * 1e14                     # per nm^2 -> per cm^2
print(f'hbar vF = {hv:.4f} eV nm;  D(E) = {c:.4f} |E| states/(eV nm^2);  {cm2(atoms):.3e} carbon atoms per cm^2')

## 1 · Counting electrons

With periodic boundaries on an $L\times L$ sheet, $\mathbf k=\tfrac{2\pi}{L}(m_x,m_y)$, so each allowed $\mathbf k$ owns an area $4\pi^2/L^2$. Counting the points inside the Fermi circle, four electrons each, gives $N\approx g\,\pi k_F^2/(4\pi^2/L^2)$ and $n=k_F^2/\pi$. The count converges to the formula as $L$ grows, and its derivative is the density of states.

In [ ]:
def count(L, kF):
    s = 2 * np.pi / L
    M = int(kF / s) + 1
    m = np.arange(-M, M + 1)
    MX, MY = np.meshgrid(m, m)
    return int(np.sum(s * s * (MX ** 2 + MY ** 2) < kF * kF))

Ef = 0.1
kF = Ef / hv
for L in (40, 100, 300, 1000, 3000):
    N = count(L, kF)
    print(f'L = {L:5d} nm: {N:7d} k-points inside, formula {kF ** 2 * L ** 2 / (4 * np.pi):10.1f}, ratio {N / (kF ** 2 * L ** 2 / (4 * np.pi)):.4f}')

# the density of states from a large sheet: count every state below E and differentiate
L = 2000
E = np.linspace(0.005, 0.3, 60)
N = np.array([4 * count(L, x / hv) for x in E])
D_counted = np.gradient(N, E) / L ** 2
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
Es = np.linspace(0.005, 0.4, 400)
axs[0].plot(Es, [4 * count(100, x / hv) for x in Es], label='counted, L = 100 nm')
axs[0].plot(Es, (Es / hv) ** 2 * 100 ** 2 / np.pi, '--', label='$k_F^2L^2/\\pi$')
axs[0].set(xlabel='$E_F$ (eV)', ylabel='electrons in the sheet'); axs[0].legend()
axs[1].plot(E, D_counted, 'o', ms=3, label='counted, L = 2000 nm')
axs[1].plot(E, c * E, label='$2|E|/\\pi\\hbar^2v_F^2$')
axs[1].set(xlabel='E (eV)', ylabel='D (states / eV nm$^2$)'); axs[1].legend()
plt.tight_layout(); plt.show()

## 2 · Electrons, holes and temperature

$n=c\,(k_BT)^2F_1(E_F/k_BT)$ and $p=c\,(k_BT)^2F_1(-E_F/k_BT)$, with the complete Fermi–Dirac integral $F_1(x)=\int_0^\infty u\,du/(1+e^{u-x})=-\mathrm{Li}_2(-e^x)$. SciPy's `spence` gives the dilogarithm, $\mathrm{Li}_2(z)=\mathrm{spence}(1-z)$. The figure repeats the video's plots at 0 K and 1000 K. Because $F_1(x)+F_1(-x)=\pi^2/6+x^2/2$, the Sommerfeld formula is exactly $n+p$.

In [ ]:
def F1(x):
    # -Li2(-e^x), with Li2(z) = spence(1 - z); for x > 0 use F1(x) = pi^2/6 + x^2/2 - F1(-x), which avoids overflow
    x = np.asarray(x, float)
    ax = np.abs(x)
    # F1(-|x|): spence loses precision as its argument nears 1 (|x| large), so use the series e^-u - e^-2u/4 + e^-3u/9 there
    series = np.exp(-ax) - np.exp(-2 * ax) / 4 + np.exp(-3 * ax) / 9
    neg = np.where(ax > 15, series, -np.real(spence(1 + np.exp(-np.minimum(ax, 15)))))
    return np.where(x > 0, np.pi ** 2 / 6 + x * x / 2 - neg, neg)
def n_of(mu, T):
    if T == 0: return c * max(mu, 0) ** 2 / 2
    t = kT(T); return c * t * t * F1(mu / t)
def p_of(mu, T): return n_of(-mu, T)
def n_quad(mu, T):
    t = kT(T)
    return quad(lambda E: c * E / (1 + np.exp(min((E - mu) / t, 700))), 0, max(mu, 0) + 60 * t, limit=300, epsrel=1e-12)[0]

err = max(abs(n_of(mu, T) / n_quad(mu, T) - 1) for mu in (-0.2, -0.05, 0, 0.05, 0.3) for T in (77, 300, 1000))
print(f'dilogarithm form vs direct integration, both branches of F1: largest relative difference {err:.1e}')
assert err < 1e-9
# the identity F1(x) + F1(-x) = pi^2/6 + x^2/2, tested on the integrals themselves (not on F1, which uses it)
F1q = lambda x: quad(lambda u: u / (1 + np.exp(min(u - x, 700))), 0, max(x, 0) + 60, points=[x] if x > 0 else None, limit=300, epsrel=1e-13)[0]
idt = max(abs(F1q(x) + F1q(-x) - (np.pi ** 2 / 6 + x * x / 2)) for x in (0, 0.7, 3, 12))
print(f'F1(x) + F1(-x) - (pi^2/6 + x^2/2), by integration: {idt:.1e}')
assert idt < 1e-9

mus = np.linspace(-0.5, 0.5, 201)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, T in zip(axs, (0, 1000)):
    ax.plot(mus, [cm2(p_of(m, T)) for m in mus], 'b', label='p')
    ax.plot(mus, [cm2(n_of(m, T)) for m in mus], 'r', label='n')
    if T: ax.plot(mus, [cm2(m * m / (np.pi * hv ** 2) + np.pi * kT(T) ** 2 / (3 * hv ** 2)) for m in mus], '--', color='orange', label='Sommerfeld = n + p')
    ax.set(xlabel='$E_F$ (eV)', title=f'T = {T} K'); ax.legend()
axs[0].set_ylabel('carrier density (cm$^{-2}$)')
plt.tight_layout(); plt.show()
for T in (300, 1000):
    print(f'neutral graphene at {T} K: n = p = {cm2(n_of(0, T)):.3e} cm^-2 = pi (kT)^2 / 6 hbar^2 vF^2 = {cm2(np.pi * kT(T) ** 2 / (6 * hv ** 2)):.3e}')

## 3 · Gating and quantum capacitance

The gate equation $e(n-p)=C_{ox}(V-E_F/e)$ has one solution for $E_F$, since its left side rises with $E_F$ and its right side falls. The quantum capacitance $C_{qm}=e^2\,d(n-p)/dE_F=e^2\frac{2k_BT}{\pi\hbar^2v_F^2}\ln[2+2\cosh(E_F/k_BT)]$, and the measured capacitance $C_{tot}=e\,d(n-p)/dV$ is $C_{ox}$ and $C_{qm}$ in series. Capacitances are per unit area (F/m²; 1 µF/cm² = 0.01 F/m²).

In [ ]:
def net(mu, T): return n_of(mu, T) - p_of(mu, T)
def solve(V, T, Cox):
    g = lambda mu: e * net(mu, T) * 1e18 - Cox * (V - mu)
    return brentq(g, -abs(V) - 1, abs(V) + 1, xtol=1e-15)
def Cqm(mu, T):
    if T == 0: return e * c * abs(mu) * 1e18
    t = kT(T); return e * c * t * np.log(2 + 2 * np.cosh(mu / t)) * 1e18
Cox = lambda d_nm, er: epsilon_0 * er / (d_nm * 1e-9)

# the closed form for C_qm against a numerical derivative of n - p
for mu, T in [(0, 300), (0.1, 300), (-0.2, 77)]:
    h = 1e-6
    num = e * (net(mu + h, T) - net(mu - h, T)) / (2 * h) * 1e18
    print(f'C_qm(E_F={mu}, {T} K) = {Cqm(mu, T) * 100:.4f} uF/cm^2, numerical {num * 100:.4f}')
    assert abs(Cqm(mu, T) / num - 1) < 1e-6
print(f'video slide: C_ox = {Cox(300, 3.9):.3e} (300 nm), {Cox(3, 3.9):.3e} (3 nm); C_qm = {Cqm(0.01, 0):.3e} (0.01 eV), {Cqm(0.1, 0):.3e} (0.1 eV) F/m^2')

# the total capacitance e d(n - p)/dV equals the series combination
C3 = Cox(3, 3.9)
for V in (0.0, 0.5, 1.0):
    h = 1e-5
    Ctot_num = e * (net(solve(V + h, 300, C3), 300) - net(solve(V - h, 300, C3), 300)) / (2 * h) * 1e18
    q = Cqm(solve(V, 300, C3), 300)
    print(f'V = {V} V: C_tot = {Ctot_num * 100:.4f} uF/cm^2, series formula {C3 * q / (C3 + q) * 100:.4f}')
    assert abs(Ctot_num / (C3 * q / (C3 + q)) - 1) < 1e-5

fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
vrange = lambda C: min(80, max(0.5, float(f'{e * 1.2e17 / C:.2g}')))      # the lab's range: about 1e13 cm^-2 for a metal, at most 80 V
for (d, er), col in zip([(300, 3.9), (10, 25), (3, 3.9)], ['C0', 'C1', 'C2']):
    vmax = vrange(Cox(d, er))
    Vs = np.linspace(-vmax, vmax, 160)              # an even count, so V = 0 itself is not on the grid
    efs = np.array([solve(v, 300, Cox(d, er)) for v in Vs])
    axs[0].plot(Vs / vmax, efs / Vs, col, label=f'{d} nm, $\\varepsilon_r$ = {er}')
    axs[1].semilogy(Vs / vmax, [Cqm(m, 300) * 100 for m in efs], col, ls=':')
    axs[1].semilogy(Vs / vmax, [Cox(d, er) * Cqm(m, 300) / (Cox(d, er) + Cqm(m, 300)) * 100 for m in efs], col)
axs[0].set(xlabel='V / V$_{max}$', ylabel='share of V in the Fermi level, $E_F/eV$'); axs[0].legend()
axs[1].set(xlabel='V / V$_{max}$', ylabel='µF/cm$^2$ (solid: $C_{tot}$, dotted: $C_{qm}$)')
plt.tight_layout(); plt.show()

## Check the lab's answers

These asserts confirm the numbers in the lab's "Predict first" answers.

In [ ]:
# Q1: 100 nm sheet, E_F = 0.1 eV: formula 73.5 electrons (18.4 k-points x 4), counted 21 x 4 = 84; n = 7.3e11 cm^-2, ~2 per 10,000 atoms
kF = 0.1 / hv
assert abs(kF - 0.152) < 0.001 and abs(kF ** 2 * 100 ** 2 / np.pi - 73.5) < 0.1 and count(100, kF) == 21
assert abs(cm2(kF ** 2 / np.pi) / 7.35e11 - 1) < 0.01 and abs(kF ** 2 / np.pi / atoms * 1e4 - 1.9) < 0.1
# Q2: doubling E_F quadruples n, to 2.9e12 cm^-2
assert abs(n_of(0.2, 0) / n_of(0.1, 0) - 4) < 1e-12 and abs(cm2(n_of(0.2, 0)) / 2.94e12 - 1) < 0.01
assert 4 * count(100, 0.1 / hv) == 84 and 4 * count(100, 0.2 / hv) == 276      # the counted 100 nm sheet: x3.3
# Q3: E_F = -0.37 eV at 0 K: p = 1.0e13 cm^-2, 2.6 per 1000 atoms
assert abs(cm2(p_of(-0.37, 0)) / 1.0e13 - 1) < 0.01 and abs(p_of(-0.37, 0) / atoms * 1000 - 2.6) < 0.05
# Q4: neutral graphene: 8.1e10 cm^-2 at 300 K and 9.0e11 at 1000 K, each of n and p
assert abs(cm2(n_of(0, 300)) / 8.08e10 - 1) < 0.005 and abs(cm2(n_of(0, 1000)) / 8.97e11 - 1) < 0.005
# Q5: the Sommerfeld formula is exactly n + p (tested with n and p integrated directly), so at E_F = 0 it is twice n
for mu, T in [(0, 300), (0.1, 300), (-0.25, 1000), (0.02, 77)]:
    somm = mu ** 2 / (np.pi * hv ** 2) + np.pi * kT(T) ** 2 / (3 * hv ** 2)
    assert abs((n_quad(mu, T) + n_quad(-mu, T)) / somm - 1) < 1e-9
assert abs(np.pi * kT(300) ** 2 / (3 * hv ** 2) / n_of(0, 300) - 2) < 1e-12
# Q6: 300 nm SiO2 at 10 V: C_ox = 11.5 nF/cm^2; a metal would hold 7.2e11, graphene 7.1e11 cm^-2; E_F = 0.087 eV (300 K) / 0.098 eV (0 K), ~1% of V
C300 = Cox(300, 3.9)
assert abs(C300 * 1e5 - 11.5) < 0.05 and abs(C300 * 10 / e * 1e-4 / 7.2e11 - 1) < 0.01
for T, ef in [(300, 0.087), (0, 0.098)]:
    mu = solve(10, T, C300); assert abs(mu - ef) < 0.001 and abs(cm2(net(mu, T)) / 7.1e11 - 1) < 0.01 and mu / 10 < 0.01
# Q7: 3 nm SiO2 at 1 V, 0 K: E_F = 0.27 eV, n = 5.3e12 (metal 7.2e12, 73%); C_qm = 6.3, C_tot = 0.97 uF/cm^2 (85% of C_ox)
mu = solve(1, 0, C3)
assert abs(mu - 0.268) < 0.001 and abs(cm2(net(mu, 0)) / 5.26e12 - 1) < 0.005 and abs(C3 * 1 / e * 1e-4 / 7.18e12 - 1) < 0.005
assert abs(Cqm(mu, 0) * 100 - 6.3) < 0.05 and abs(C3 * Cqm(mu, 0) / (C3 + Cqm(mu, 0)) / C3 - 0.85) < 0.01
# Q8: at the Dirac point, 300 K: C_qm = 0.84 uF/cm^2; with C_ox = 1.15 the total is 0.49 (42% of C_ox); at 0 K both vanish
q = Cqm(0, 300)
assert abs(q * 100 - 0.844) < 0.001 and abs(C3 * q / (C3 + q) * 100 - 0.487) < 0.001 and Cqm(0, 0) == 0
print('all the lab\'s answers check out')

## Things to try

1. **Charged impurities.** Real graphene has electron–hole puddles: add a random potential, so that $E_F$ varies across the sheet with a spread of tens of meV. How does it round off the dip in $C_{qm}$?
2. **A dual gate.** Put graphene between a top and a back gate, $e(n-p)=C_t(V_t-E_F/e)+C_b(V_b-E_F/e)$. Which combinations of $V_t$ and $V_b$ keep graphene neutral?
3. **Bilayer graphene.** Its bands near zero energy are parabolic, so $D$ is constant. Redo the gate calculation and compare the quantum capacitance.
4. **Where the cones end.** Replace $D(E)=2|E|/\pi\hbar^2v_F^2$ by the tight-binding density of states from the graphene band-structure lab. At what gate voltage does it matter for a 3 nm oxide?